### Evaluating the Amount of Training Data Required For a Char-RNN to Succesfully Produce Shakespeare Text

#### Cogs 181 Final Project Notebook

Name: Ankita Inamti 

PID: A17388687


In [15]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import time

#### Load and encode the data

We are using tiny shakespeare dataset

In [16]:
with open("tiny-shakespeare.txt", "r") as f:
    text = f.read()

chars = sorted(set(text))
VOCAB = len(chars)
ch2i = {c: i for i, c in enumerate(chars)}
i2ch = {i: c for i, c in enumerate(chars)}

def encode(s): return [ch2i[c] for c in s]
def decode(l): return "".join(i2ch[i] for i in l)

ids = torch.tensor(encode(text), dtype=torch.long)
print(f"Total chars: {len(text):,} | Vocab size: {VOCAB}")

Total chars: 1,115,394 | Vocab size: 65


#### Hyperparameters

Explain

In [17]:
HIDDEN_SIZE   = 256
SEQ_LEN       = 100
BATCH_SIZE    = 64
LEARNING_RATE = 3e-4
EPOCHS        = 5
FRACTIONS     = [0.10, 0.25, 0.50, 1.00]
SAMPLE_LEN    = 500
DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using: {DEVICE}")

Using: cpu


In [18]:
from torch.utils.data import Dataset, DataLoader

class CharDataset(Dataset):
    def __init__(self, data, seq_len):
        self.data    = data
        self.seq_len = seq_len

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, i):
        x = self.data[i     : i + self.seq_len]
        y = self.data[i + 1 : i + self.seq_len + 1]
        return x, y

def make_loader(ids, fraction):
    subset = ids[:int(len(ids) * fraction)]
    ds = CharDataset(subset, SEQ_LEN)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

In [19]:
class CharRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.embed  = nn.Embedding(vocab_size, hidden_size)
        self.lstm   = nn.LSTM(hidden_size, hidden_size, num_layers=2,
                              dropout=0.3, batch_first=True)
        self.fc     = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, state=None):
        x = self.embed(x)
        out, state = self.lstm(x, state)
        logits = self.fc(out)
        return logits, state

    def init_state(self, batch_size):
        h = torch.zeros(2, batch_size, self.hidden_size).to(DEVICE)
        c = torch.zeros(2, batch_size, self.hidden_size).to(DEVICE)
        return (h, c)

#### Training function

In [20]:
def train(fraction):
    loader = make_loader(ids, fraction)
    model  = CharRNN(VOCAB, HIDDEN_SIZE).to(DEVICE)
    opt    = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    loss_fn = nn.CrossEntropyLoss()

    history = []
    for epoch in range(EPOCHS):
        model.train()
        total_loss, steps = 0, 0
        t0 = time.time()

        for x, y in loader:
            x, y  = x.to(DEVICE), y.to(DEVICE)
            state = model.init_state(x.size(0))
            opt.zero_grad()
            logits, _ = model(x, state)
            loss = loss_fn(logits.reshape(-1, VOCAB), y.reshape(-1))
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)  # prevent exploding gradients
            opt.step()
            total_loss += loss.item()
            steps += 1

        avg_loss = total_loss / steps
        history.append(avg_loss)
        print(f"  [Epoch {epoch+1}/{EPOCHS}] loss={avg_loss:.4f} ({time.time()-t0:.1f}s)")

    return model, history

#### Sampling function

In [21]:
@torch.no_grad()
def sample(model, seed_text, length=SAMPLE_LEN, temperature=0.8):
    model.eval()
    ids_in = torch.tensor(encode(seed_text), dtype=torch.long).unsqueeze(0).to(DEVICE)
    state  = model.init_state(1)
    # warm up on seed
    _, state = model(ids_in, state)
    generated = list(seed_text)
    x = ids_in[:, -1:]
    for _ in range(length):
        logits, state = model(x, state)
        logits = logits[0, -1] / temperature
        probs  = torch.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, 1).item()
        generated.append(i2ch[next_id])
        x = torch.tensor([[next_id]], dtype=torch.long).to(DEVICE)
    return "".join(generated)

In [ ]:
results = {}

for frac in FRACTIONS:
    n_chars = int(len(text) * frac)
    print(f"\n{'='*50}")
    print(f"Training on {frac*100:.0f}% of data ({n_chars:,} chars)")
    print(f"{'='*50}")
    model, history = train(frac)
    results[frac] = {"model": model, "history": history}


Training on 10% of data (111,539 chars)
